## 💡 Worked Example: Simplest Possible Message Passing

Before implementing the full EGNN, here's the simplest possible GNN layer:

```python
# Simplest message passing: average neighbor features
def simple_message_pass(node_features, edge_index):
    # edge_index[0] = source nodes, edge_index[1] = target nodes
    n_nodes = node_features.shape[0]
    new_features = torch.zeros_like(node_features)
    counts = torch.zeros(n_nodes)
    for src, tgt in edge_index.T:
        new_features[tgt] += node_features[src]  # aggregate
        counts[tgt] += 1
    return new_features / counts.unsqueeze(1).clamp(min=1)  # mean
```

The EGNN below adds: (1) learnable weights, (2) coordinate updates, (3) edge features.

# Graph Neural Networks — Deep Dive
## From Message Passing Theory to Equivariant Architectures

**TL;DR (Plain English):** Graph Neural Networks let you do machine learning on graph-structured data — molecules, protein structures, interaction networks. Unlike images (regular grids) or sequences (1D arrays), graphs have irregular connectivity. GNNs learn by passing messages between connected nodes, aggregating neighborhood information iteratively. This notebook covers the full spectrum: theoretical foundations (WL isomorphism test), classic architectures (GCN, GAT, GraphSAGE, GIN), molecular applications (MPNN), and modern equivariant GNNs (EGNN, SE(3)-Transformer) used in AlphaFold3 and drug discovery.

**After this notebook you can:**
- Implement any GNN architecture from scratch in PyTorch
- Choose the right GNN for a given task (molecular property, protein interaction, knowledge graph)
- Understand equivariance and why it matters for 3D molecular data
- Explain GNN limitations (over-smoothing, over-squashing, expressivity limits)
- Use PyTorch Geometric (PyG) efficiently for large-scale graphs

## GNN Deep Dive — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **MPNN (Message Passing Neural Network)** | General GNN framework: message -> aggregate -> update — most GNNs are special cases |
| **readout** | Global pooling that converts node-level representations to a graph-level representation |
| **graph-level prediction** | Single output for the whole graph (e.g. protein stability) — needs readout |
| **node-level prediction** | Output per node (e.g. per-residue pLDDT) — no readout needed |
| **over-smoothing** | Problem with deep GNNs: all node representations converge to the same vector after many layers |
| **SE(3)-equivariant GNN** | GNN whose outputs transform correctly under 3D rotations and translations |
| **EGNN** | Equivariant GNN that updates both node features AND 3D coordinates in each layer |
| **SchNet** | GNN for molecules using radial basis functions to encode interatomic distances |
| **DimeNet** | GNN that uses both distances AND angles — captures more geometric information |
| **positional encoding** | Extra feature encoding a node's position in the graph (for sequential data like proteins) |
| **batch normalisation** | Normalises layer inputs across a batch — stabilises training of deep GNNs |
| **residual connection** | Skip connection: `output = layer(x) + x` — prevents vanishing gradients in deep networks |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students comfortable with deep learning who want to reason about relational and geometric structure.

**How to study it on a first pass:**
- understand graph construction before architecture variants
- keep invariance/equivariance intuitive before making it formal
- connect node/edge features to real protein structure meaning

**Common traps:** thinking a graph is just an adjacency matrix, skipping feature design, and treating geometric ML as a bag of acronyms.


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Protein graphs** — residue nodes, distance-based edges, node/edge features. *Review: `06_structural_ml_gnns/01_structure_ml.ipynb`*
- **Message passing** — aggregate neighbor features to update each node. *Review: `06/01`*
- **PyTorch nn.Module** — defining layers and forward methods. *Review: `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`*
- **3D coordinates** — PDB format, Cα positions, RMSD. *Review: `03_protein_structural_biology/01_structure_analysis.ipynb`*

**Quick recap of terms used here:**
- **equivariance** — rotating input rotates output by the same amount (for vector quantities)
- **invariance** — output doesn't change when input is rotated (for scalar quantities like energy)
- **SchNet** — GNN using radial basis functions to encode interatomic distances
- **EGNN** — updates both features AND 3D coordinates equivariantly

## 📈 Difficulty Progression in This Notebook

| Section | Level | What you'll learn |
|---------|-------|-------------------|
| Message Passing basics | 🟢 Basic | What a GNN layer does — aggregate neighbor features |
| EGNN implementation | 🟡 Intermediate | Equivariant updates to both features AND coordinates |
| SE(3)-equivariance proof | 🔴 Advanced | Why rotating input rotates output by the same amount |
| SchNet / DimeNet comparison | 🔴 Advanced | Radial basis functions vs angle-aware message passing |

**Recommendation:** If you're new to GNNs, focus on 🟢 and 🟡 sections first. Come back to 🔴 after Module 07.

## Canonical Resource Upgrade

Useful anchors:
- [Stanford CS224W](https://web.stanford.edu/class/cs224w/)
- [Graph Representation Learning](https://www.cs.mcgill.ca/~wlh/grl_book/)
- [PyTorch Geometric docs](https://pytorch-geometric.readthedocs.io/)


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 00/06 PyTorch Fundamentals + 06/01 Structure ML (GNN intro) |
| You Are Here | Module 06/02 — GNN Deep Dive (dedicated theory + architecture notebook) |
| Next Steps | 07/01 AlphaFold3 Architecture (uses IPA, a GNN variant) → 10/01 Fine-tuning Capstone |
| Goal | Implement GCN/GAT/GIN/EGNN from scratch; use PyG for production graphs |

### From Scratch? Start Here:
1. **Step 1:** [3Blue1Brown — Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) (matrix operations for graph Laplacian)
2. **Step 2:** [CS224W Lecture 1-3](https://www.youtube.com/playlist?list=PLoROMvodv4rPLKxIpqhjhPgdQy7imNkDn) — graph ML intro (free YouTube)
3. **Step 3:** 06/01 Structure ML notebook (GNN intro with protein graphs)
4. **Step 4:** This notebook — full architecture deep dive

**Time:** 15–25 hours | **Difficulty:** 5/5

### Cross-References
- Builds on: 06/01 (GNN intro), 00/06 (PyTorch), 03/01 (protein structures as graphs)
- Used by: 07/01 (AF3 uses IPA — Invariant Point Attention, a GNN variant), 10/01 (graph-based mutation effect prediction)
- Related: 05/01 covers transformers — Graph Transformers bridge GNNs and attention

---
## Section 1 — Graph Theory Foundations

### 1.1 Graph Representations

A graph `G = (V, E)` has nodes `V` and edges `E`. Three standard representations:

| Representation | Storage | Access time | Best for |
|---|---|---|---|
| Adjacency matrix `A` | O(N²) | O(1) | Dense graphs, spectral methods |
| Edge list `(src, dst)` | O(E) | O(E) | Sparse graphs, GNN message passing |
| Adjacency list | O(N+E) | O(deg) | General purpose |

### 1.2 Graph Laplacian

The **graph Laplacian** `L = D - A` where `D` is the degree matrix. Properties:
- Symmetric positive semi-definite
- `L·1 = 0` (constant signal is in null space)
- Eigenvalues `0 = λ₁ ≤ λ₂ ≤ ... ≤ λₙ` encode graph structure
- **Fiedler value** `λ₂` = algebraic connectivity (larger = more connected)
- GCN is a 1st-order Chebyshev polynomial approximation of spectral convolution on `L`

### 1.3 Weisfeiler-Leman (WL) Isomorphism Test

The **1-WL test** is the theoretical ceiling for GNN expressivity:
1. Assign each node a color (feature hash)
2. Iteratively update: new color = hash(current color, multiset of neighbor colors)
3. If two graphs always produce the same color histogram — they look identical to any GNN using sum/mean/max aggregation

**Key result (Xu et al. 2019):** Standard message-passing GNNs are at most as powerful as 1-WL. GIN achieves this upper bound. To go beyond 1-WL you need higher-order GNNs (k-WL, expensive) or structural features (cycle counts, node distances).

### 📦 Libraries Used Here

- **`torch.nn`** — Neural network layers: Linear (matrix multiply), ReLU (activation), LayerNorm (stabiliser).
- **`torch.nn.functional as F`** — Functions without learnable parameters: softmax, relu, normalise.
- **`torch_geometric`** — Extension of PyTorch for graph-structured data (proteins as graphs).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data

# Generic GNN cell
torch.manual_seed(42)
from torch_geometric.nn import GCNConv, global_mean_pool

n = 20
x = torch.randn(n, 16)
edge_index = torch.randint(0, n, (2, 60))
batch = torch.zeros(n, dtype=torch.long)

conv = GCNConv(16, 32)
h = F.relu(conv(x, edge_index))
pooled = global_mean_pool(h, batch)
print(f"Node features: {x.shape} -> {h.shape}")
print(f"Graph embedding: {pooled.shape}")

> **Expected output:** Node feature dimensions before and after message passing (e.g., `[N, 3] -> [N, 64]`) and a graph-level embedding shape (e.g., `[1, 64]`)  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

---
## Section 2 — GCN, GraphSAGE, GIN from Scratch

### 2.1 Graph Convolutional Network (GCN) — Kipf & Welling 2017

**Core idea:** Spectral graph convolution, approximated as a localized 1st-order filter.

**Update rule:**
```
H^{l+1} = sigma( D^{-1/2} (A + I) D^{-1/2}  H^l  W^l )
         = sigma(          A_hat              H^l  W^l )
```
- `A + I`: add self-loops so each node includes its own features
- `D^{-1/2} ... D^{-1/2}`: symmetric normalization prevents scale explosion with depth
- Single linear transform `W` shared across all nodes (no node-specific params)

**Limitations:** Transductive (requires full graph at train time), isotropic aggregation (ignores edge types), power limited by 1-WL.

### 2.2 GraphSAGE — Hamilton et al. 2017

**Key innovation:** *Inductive* learning — can generalize to unseen nodes at test time by learning an aggregation function, not graph-specific weights.

**Update rule:**
```
h_N(i) = AGGREGATE({h_j : j in N(i)})
h_i'   = sigma( W · concat(h_i, h_N(i)) )
```
Concatenating self + neighbor (vs GCN's sum) preserves the distinction between a node and its neighborhood.

### 2.3 Graph Isomorphism Network (GIN) — Xu et al. 2019

**Theoretical motivation:** Sum aggregation + MLP (not linear) is provably as expressive as the 1-WL test — the maximum any message-passing GNN can achieve.

**Update rule:**
```
h_i' = MLP( (1 + epsilon) * h_i  +  sum_{j in N(i)} h_j )
```
- **MLP** (not linear `W`) is the key — linear maps cannot distinguish sum-equal multisets
- **epsilon** is a learnable parameter that weights self vs neighbor contribution
- **Sum** (not mean/max) preserves multiset information (mean/max lose count info)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data

# Generic GNN cell
torch.manual_seed(42)
from torch_geometric.nn import GCNConv, global_mean_pool

n = 20
x = torch.randn(n, 16)
edge_index = torch.randint(0, n, (2, 60))
batch = torch.zeros(n, dtype=torch.long)

conv = GCNConv(16, 32)
h = F.relu(conv(x, edge_index))
pooled = global_mean_pool(h, batch)
print(f"Node features: {x.shape} -> {h.shape}")
print(f"Graph embedding: {pooled.shape}")

---
## Section 3 — Graph Attention Network (GAT & GATv2)

### 3.1 GAT v1 — Veličković et al. 2018

**Core idea:** Not all neighbors are equally important — learn attention weights.

**Attention mechanism:**
```
e_ij = LeakyReLU( a^T · [W·h_i || W·h_j] )
alpha_ij = softmax_j( e_ij )   # normalize over incoming edges per node
h_i' = sigma( sum_j  alpha_ij · W · h_j )
```

**Multi-head:** Run K independent attention heads, concatenate (or average) outputs.

**Problem — static attention (Brody et al. 2021):**
```
e_ij = a^T · [W·h_i || W·h_j]
     = a_l^T · W·h_i  +  a_r^T · W·h_j     (linear decomposition!)
     = f(i)            +  g(j)
```
Because `LeakyReLU` is applied *after* the linear projection, the score decomposes as `f(i) + g(j)`. This means for a fixed node `i`, the ranking of neighbors `j` is the same regardless of `i`'s features — attention is **static**.

### 3.2 GATv2 — Brody et al. 2021

**Fix:** Apply `LeakyReLU` *before* the final projection:
```
e_ij = a^T · LeakyReLU( W · [h_i || h_j] )    # GATv2 — nonlinearity FIRST
```
Now `W · [h_i || h_j]` creates an entangled representation that *cannot* be decomposed as `f(i) + g(j)`. Attention becomes **dynamic** — node `i`'s context genuinely changes which neighbor `j` gets attended to.

In [ ]:
# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, Batch

# --- ProteinGAT class definition ---
class ProteinGAT(nn.Module):
    """Graph Attention Network for protein function prediction."""
    # --- __init__() function ---
    def __init__(self, in_channels=20, hidden=64, heads=4, n_classes=2):
        super().__init__()
        # Set gat1
        self.gat1 = GATConv(in_channels, hidden, heads=heads, concat=True, dropout=0.1)
        # Set gat2
        self.gat2 = GATConv(hidden*heads, hidden, heads=1, concat=False, dropout=0.1)
        # Set bn1
        self.bn1 = nn.BatchNorm1d(hidden*heads)
        # Set bn2
        self.bn2 = nn.BatchNorm1d(hidden)
        # Set classifier
        self.classifier = nn.Linear(hidden, n_classes)

    # --- forward() function ---
    def forward(self, x, edge_index, batch):
        x = F.elu(self.bn1(self.gat1(x, edge_index)))
        x = F.elu(self.bn2(self.gat2(x, edge_index)))
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# Compare GCN vs GAT
torch.manual_seed(42)
from torch_geometric.nn import GCNConv

# Create test data
n_nodes, n_features = 50, 20
x = torch.randn(n_nodes, n_features)
edge_index = torch.randint(0, n_nodes, (2, 150))
# Initialize tensor
batch = torch.zeros(n_nodes, dtype=torch.long)

model_gat = ProteinGAT(in_channels=20, hidden=32, heads=4, n_classes=2)
out = model_gat(x, edge_index, batch)
print(f"GAT output: {out.shape}")
print(f"GAT parameters: {sum(p.numel() for p in model_gat.parameters()):,}")
print()
print("GAT advantage: learns WHICH neighbors to attend to")
print("For protein graphs: attention ≈ residue interaction importance")

---
## Section 4 — Equivariant GNNs (EGNN)

### 4.1 Why Equivariance Matters for 3D Molecules

Protein and molecular structures exist in 3D space. Physical properties (energy, forces, binding affinity) must be **invariant** or **equivariant** to rotation, translation, and reflection:

| Property | Requirement |
|---|---|
| Molecular energy | **Invariant** — same energy regardless of orientation |
| Atomic forces | **Equivariant** — forces rotate with the molecule |
| Protein coordinates | **Equivariant** — positions rotate when molecule rotates |

**The symmetry group:** E(3) = rotations + translations + reflections (Euclidean group in 3D). SE(3) drops reflections (special Euclidean = proper rigid motions).

### 4.2 Approaches to Equivariance

| Method | Key idea | Complexity | Examples |
|---|---|---|---|
| Invariant features only | Use distances, angles — discard direction | Simple | SchNet, DimeNet |
| EGNN | Equivariant coordinate updates via displacement vectors | Low | EGNN (Satorras 2021) |
| Tensor field networks | Irreducible representations (spherical harmonics) | High | TFN, SE(3)-Transformer |
| Equiformer | SO(3)-equivariant transformer | Very high | EquiformerV2 (SOTA OC20) |

### 4.3 EGNN Core Insight

**Key trick:** Use `||x_i - x_j||²` as an edge feature. This squared distance is:
- Rotation-invariant (distances don't change under rotation)
- Translation-invariant (relative positions are unaffected)

Then update coordinates **equivariantly** by weighting the displacement vector `(x_i - x_j)`:
```
x_i' = x_i  +  sum_j  (x_i - x_j) * phi_x(m_ij)
```
If you rotate the input positions `x → Rx`, the displacements also rotate `(x_i - x_j) → R(x_i - x_j)`, so the output rotates too — equivariance by construction.

### 📖 Reading Guide — EGNN — E(3)-Equivariant Graph Neural Network Layer

`class EGNNLayer(nn.Module):`
→ *Define a new type of neural network layer. nn.Module = the base class for all PyTorch layers.*

`def __init__(self, ...):`
→ *__init__ runs once when you create the layer — it defines what learnable weights it has.*

`def forward(self, h, x, edge_index):`
→ *forward() runs every time data passes through the layer. h=node features, x=3D positions, edge_index=which nodes are connected.*

`row, col = edge_index`
→ *Split the edge list into source nodes (row) and target nodes (col).*

`rel_pos = x[row] - x[col]`
→ *Relative position vector: how far is each neighbour, and in what direction?*

`dist = rel_pos.norm(dim=-1, keepdim=True)`
→ *Euclidean distance between connected residues. norm() = sqrt(x²+y²+z²).*

`msg = self.msg_net(torch.cat([h[row], h[col], dist], dim=-1))`
→ *Message: combine source features + target features + distance into one vector. cat = concatenate.*

`agg = scatter_mean(msg, col, dim=0)`
→ *Aggregation: average all incoming messages for each node. A node with 5 neighbours gets the average of 5 messages.*

`h_new = self.update_net(torch.cat([h, agg], dim=-1))`
→ *Update: combine old features + aggregated messages to get new features.*

`x_new = x + delta_x`
→ *Move each atom's 3D position by a small learnable offset. This is the equivariant part.*



In [ ]:
# --- Imports ---
import torch
import torch.nn as nn

# --- EGNNLayer class definition ---
class EGNNLayer(nn.Module):
    """E(n) Equivariant Graph Neural Network layer (Satorras et al. 2021)."""
    # --- __init__() function ---
    def __init__(self, d_in, d_hidden=64):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(d_in * 2 + 1, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_hidden), nn.SiLU()
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(d_in + d_hidden, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_in)
        )
        # Set coord_mlp
        self.coord_mlp = nn.Sequential(
            nn.Linear(d_hidden, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, 1)
        )

    # --- forward() function ---
    def forward(self, h, x, edge_index):
        src, dst = edge_index
        r_sq = ((x[src] - x[dst])**2).sum(-1, keepdim=True)
        # Combine tensors
        e_input = torch.cat([h[src], h[dst], r_sq], dim=-1)
        m = self.edge_mlp(e_input)  # edge messages

        # Equivariant coordinate update
        x_diff = x[src] - x[dst]
        # Initialize tensor
        x_agg = torch.zeros_like(x)
        x_agg.index_add_(0, dst, x_diff * self.coord_mlp(m))

        # Node update
        m_agg = torch.zeros(h.shape[0], m.shape[-1])
        m_agg.index_add_(0, dst, m)
        # Combine tensors
        h_new = h + self.node_mlp(torch.cat([h, m_agg], dim=-1))
        x_new = x + x_agg

        return h_new, x_new

torch.manual_seed(42)
N, d = 30, 16
h = torch.randn(N, d)
x = torch.randn(N, 3)
edge_index = torch.randint(0, N, (2, 90))

layer = EGNNLayer(d_in=d)
h_new, x_new = layer(h, x, edge_index)
print(f"h: {h.shape} -> {h_new.shape}")
print(f"x: {x.shape} -> {x_new.shape}")
print("EGNN key property: coordinate updates are equivariant to rotations/translations")

## 🏁 Checkpoint -- Pause and Verify

Before continuing, make sure you can:
1. Explain message passing: how node features are aggregated from neighbors
2. Implement a single GNN layer (GCN or GAT) and describe the update equations
3. Build a protein graph from 3D coordinates (nodes = residues, edges = contacts within a distance cutoff)

**If any of these feel shaky**, re-read the section above or review the prerequisite notebook listed in the Cross-References section.

---
## Section 5 — PyTorch Geometric (PyG) in Practice

### 5.1 PyG `MessagePassing` Base Class

PyG provides the `MessagePassing` base class that handles all the graph plumbing. You only override three methods:

```
forward(x, edge_index)
  └─> propagate(edge_index, x=x)
         ├─> message(x_i, x_j, edge_attr)    # compute m_ij per edge
         ├─> aggregate('sum'/'mean'/'max')    # aggregate messages per node
         └─> update(aggr_out, x)             # update node features
```

**Magic naming:** PyG uses `_i` and `_j` suffixes. If you call `propagate(edge_index, x=x)`, then in `message()`, `x_i = x[edge_index[1]]` (destination) and `x_j = x[edge_index[0]]` (source).

### 5.2 PyG `Data` Object

```python
Data(x=..., edge_index=..., edge_attr=..., y=..., pos=...)
```
- `x`: Node feature matrix `(N, F)`
- `edge_index`: Edge list `(2, E)` — COO format, source then destination
- `edge_attr`: Edge features `(E, D)`
- `y`: Labels (node-level `(N,)`, graph-level `(1,)`, or link-level `(E,)`)
- `pos`: 3D coordinates `(N, 3)` — used by spatial GNNs

### 5.3 Batching Graphs

PyG batches multiple graphs by **merging into one large disconnected graph** with offset node indices. `DataLoader` handles this automatically. The `batch` tensor tracks which nodes belong to which graph for global pooling.

> **Why this code:** We import the libraries needed for this section so all subsequent cells can use them.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data

# Generic GNN cell
torch.manual_seed(42)
from torch_geometric.nn import GCNConv, global_mean_pool

n = 20
x = torch.randn(n, 16)
edge_index = torch.randint(0, n, (2, 60))
batch = torch.zeros(n, dtype=torch.long)

conv = GCNConv(16, 32)
h = F.relu(conv(x, edge_index))
pooled = global_mean_pool(h, batch)
print(f"Node features: {x.shape} -> {h.shape}")
print(f"Graph embedding: {pooled.shape}")

---
## Section 6 — Graph Transformers & Scalability

### 6.1 Over-Smoothing

**Problem:** As GNN depth increases, node features converge to the same value — the graph signal is smoothed away.

**Why:** Repeated averaging `H' = A_hat H` is a low-pass filter. After `k` steps, node features converge to eigenvectors of `A_hat` corresponding to small eigenvalues → all features look alike.

**Mitigation strategies:**
| Strategy | How it works |
|---|---|
| Residual connections | `H' = H + GNN(H)` — preserves original signal |
| JK-Net | Concatenate representations from all layers |
| DropEdge | Randomly drop edges during training — slows smoothing |
| PairNorm | Normalize so mean pairwise distance is constant |
| APPNP | Separate feature transformation from propagation |

### 6.2 Over-Squashing

**Problem:** Exponentially many nodes in k-hop neighborhood must be compressed into a fixed-size representation.

A message from a node `v` to node `u` (k hops away) is compressed through `k` graph bottlenecks. If the graph has high curvature (many short paths), information is lost.

**Fix:** Graph rewiring (add long-range edges), virtual nodes, or use Graph Transformers with global attention.

### 6.3 Graph Transformers vs Message Passing GNNs

| | GNN | Graph Transformer |
|---|---|---|
| Receptive field | k-hop neighborhood | All nodes (global) |
| Complexity | O(E) = O(N) for sparse | O(N²) |
| Long-range | Needs many layers | Natural |
| Best for | Large sparse graphs | Smaller graphs, long-range deps |
| Examples | GCN, GAT, GIN | Graphormer, GraphGPS |

In [ ]:
# --- Imports ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import matplotlib.pyplot as plt

# --- measure_oversmoothing() function ---
def measure_oversmoothing(model, x, edge_index, batch, n_layers_list):
    """Measure feature diversity (MAD) as depth increases → over-smoothing."""
    mad_scores = []
    # --- Loop over n_layers_list ---
    for n_layers in n_layers_list:
        h = x.clone()
        conv = GCNConv(x.shape[1], x.shape[1])
        # --- Loop over range(n_layers) ---
        for _ in range(n_layers):
            h = F.relu(conv(h, edge_index))

        # Mean Average Distance (MAD) — higher = more distinct features
        h_norm = F.normalize(h, dim=-1)
        sim_matrix = h_norm @ h_norm.T
        cos_dists = 1 - sim_matrix
        mad = cos_dists[cos_dists > 0].mean().item()
        mad_scores.append(mad)
    return mad_scores

torch.manual_seed(42)
N = 50
x = torch.randn(N, 32)
edge_index = torch.randint(0, N, (2, 200))  # dense graph → stronger over-smoothing
# Initialize tensor
batch = torch.zeros(N, dtype=torch.long)

n_layers_list = [1, 2, 4, 8, 16, 32]
mad_scores = measure_oversmoothing(None, x, edge_index, batch, n_layers_list)

print("Layer depth vs feature diversity (MAD):")
# --- Loop ---
for d, mad in zip(n_layers_list, mad_scores):
    bar = '#' * int(mad * 50)
    print(f"  {d:2d} layers: MAD={mad:.4f} {bar}")

print("\nOver-smoothing: deep GNNs converge to same representation")
print("Fixes: residual connections, normalization, DropEdge, graph transformers")

---
## Resources

### 1 — Theory: Foundations and Math
- **Spectral Graph Theory:** Graph Laplacian, Chebyshev polynomials → GCN derivation
- **WL Isomorphism Test:** How it bounds GNN expressivity (Weisfeiler-Leman test, 1968)
- **Equivariance Theory:** Group theory (SE(3), E(3), O(3)), irreducible representations
- **Over-smoothing Theory:** Dirichlet energy, connection to graph heat diffusion
- Papers: [GCN (Kipf 2017)](https://arxiv.org/abs/1609.02907), [GAT (Velickovic 2018)](https://arxiv.org/abs/1710.10903), [GIN (Xu 2019)](https://arxiv.org/abs/1810.00826), [EGNN (Satorras 2021)](https://arxiv.org/abs/2102.09844), [GATv2 (Brody 2022)](https://arxiv.org/abs/2105.14491)

### 2 — Must-Have Popular Resources
#### 🟢 Start Here (Zero GNN Background)
- 🆓 **CS224W Stanford** — [youtube.com/playlist](https://www.youtube.com/playlist?list=PLoROMvodv4rPLKxIpqhjhPgdQy7imNkDn) — free YouTube, 20 lectures, start with Lecture 1-3
- 🆓 **Graph Representation Learning (Hamilton)** — [cs.mcgill.ca/~wlh/grl_book](https://www.cs.mcgill.ca/~wlh/grl_book/) — free PDF, 140 pages, readable
- 🆓 **Distill.pub GNN explainer** — [distill.pub/2021/gnn-intro](https://distill.pub/2021/gnn-intro/) — beautiful interactive visualizations
- 🆓 **PyG tutorials** — [pytorch-geometric.readthedocs.io/en/latest/get_started](https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html) — hands-on code
- **Book:** [Graph Representation Learning (William L. Hamilton)](https://www.cs.mcgill.ca/~wlh/grl_book/) — free PDF, THE GNN textbook
- **Course:** [CS224W Stanford — Machine Learning with Graphs](https://web.stanford.edu/class/cs224w/) — Jure Leskovec, free on YouTube
- **Video:** [CS224W 2023 YouTube playlist](https://www.youtube.com/playlist?list=PLoROMvodv4rPLKxIpqhjhPgdQy7imNkDn) — 20 lectures
- **GitHub:** [pyg-team/pytorch_geometric](https://github.com/pyg-team/pytorch_geometric) — 21k stars, production GNN library
- **GitHub:** [dmlc/dgl](https://github.com/dmlc/dgl) — 13k stars, Deep Graph Library (alternative to PyG)
- **HuggingFace:** [graphs-datasets collection](https://huggingface.co/datasets?task_categories=task_categories%3Agraph-ml)
- **Kaggle:** [Predicting Molecular Properties](https://www.kaggle.com/c/champs-scalar-coupling) — real GNN competition

### 3 — Practicals: Hands-On
- **Exercise 1:** Implement GCN, GraphSAGE, GIN from scratch (no PyG) — this notebook
- **Exercise 2:** Train on [OGB-MOLHIV](https://ogb.stanford.edu/docs/graphprop/) molecular property prediction
- **Exercise 3:** Implement EGNN, verify E(3) equivariance with rotation check
- GitHub: [snap-stanford/ogb](https://github.com/snap-stanford/ogb) — Open Graph Benchmark
- GitHub: [chaitjo/geometric-gnn-dojo](https://github.com/chaitjo/geometric-gnn-dojo) — equivariant GNN tutorial
- Kaggle: [OGB leaderboards](https://ogb.stanford.edu/docs/leader_graphprop/) — compare your model
- HuggingFace: [graphormer model](https://huggingface.co/microsoft/graphormer-base-pcqm4mv2)

### 4 — Real-World Problems
- **computational biology ML teams:** GNN + diffusion for small molecule generation and protein-ligand docking
- **Schrodinger:** Force field GNNs (ANI, DimeNet++) for drug design
- **Genentech/Roche:** Protein interaction network GNNs for target identification
- **Dataset:** [PLINDER](https://huggingface.co/datasets/plinder-org/plinder) — 450k protein-ligand structures on HuggingFace
- **Dataset:** [PCQM4Mv2](https://ogb.stanford.edu/docs/lsc/pcqm4mv2/) — 3.8M molecules, quantum property prediction
- **Application:** Predict protein-ligand binding affinity using structure-based GNN

### 5 — Interview Tips
- **Must know:** Why is message passing <= 1-WL? Give a concrete example of two graphs GCN can't distinguish
- **Must know:** GCN vs GAT vs GIN — expressivity ordering and when to use each
- **Must know:** What is E(3)-equivariance and why does it matter for molecular data?
- **Common mistake:** Using isotropic aggregation (sum/mean) when edge directionality matters
- **Common mistake:** Not normalizing GCN adjacency matrix — causes feature scale explosion with depth
- **Common mistake:** Using GAT v1 for tasks where edge context matters — use GATv2 instead
- **Pro tip:** computational biology ML teams interviews ask about EGNN vs GVP-GNN vs SE(3)-Transformer tradeoffs
- **Pro tip:** Know that PyG's MessagePassing.propagate() = message() -> aggregate() -> update()

### 6 — Hot Industry Topics
- **Trending:** [EquiformerV2](https://github.com/atomicarchitects/equiformer_v2) — SO(3)-equivariant, SOTA on OC20 catalyst dataset
- **Trending:** [Uni-Mol](https://github.com/deepmodeling/Uni-Mol) — unified molecular pretraining (MSRA/DP Tech)
- **Trending:** [GraphGPS](https://github.com/rampasek/GraphGPS) — GNN + Transformer hybrid, SOTA on benchmarks
- **Trending:** [Graphormer](https://github.com/microsoft/Graphormer) — Microsoft's Graph Transformer (PCQM4M winner)
- **Trending:** [Open Catalyst Project (OCP)](https://github.com/Open-Catalyst-Project/ocp) — Meta AI + CMU, GNNs for catalysis
- **Build:** Protein-ligand binding GNN on PLINDER dataset — GATv2 encoder + DimeNet++ edge features -> binding affinity regression
- **Build:** Implement EGNN from scratch, verify equivariance, train on QM9 molecular properties
- **Build:** Graph Transformer for TCR-pMHC binding prediction (connects to Module 10 capstone)

## Interview Q&A — GNNs Deep Dive

**Q: Why is GIN (Graph Isomorphism Network) more expressive than GCN?**
A: GCN uses normalized adjacency: h_v = σ(W · mean(h_u, u∈N(v))). Mean aggregation loses information about neighborhood size. GIN uses sum aggregation: h_v = MLP((1+ε)h_v + Σ h_u), which is provably as powerful as the Weisfeiler-Lehman graph isomorphism test (1-WL). Mean/max aggregation are strictly less expressive.

**Q: What is over-smoothing in GNNs and how do you detect it?**
A: With many GNN layers, node representations become indistinguishable — all nodes converge to the same vector. Detected by: Dirichlet energy = Σ_{(u,v)∈E} ||h_u - h_v||² → near 0. Fixes: (1) residual connections (JK-Net), (2) limit to 2-4 layers, (3) graph transformers (global attention bypasses stacking).

**Q: What makes EGNN equivariant and why does this matter for proteins?**
A: EGNN updates positions: x_i ← x_i + Σ (x_i - x_j)·φ_x(m_ij). Messages depend only on distances (||x_i - x_j||), not absolute coordinates. Equivariant means: if you rotate/translate the protein, the model gives the same predictions (in the new frame). For proteins: two identical structures rotated differently should give same stability prediction.

**Q: Explain the difference between GATv1 and GATv2.**
A: GATv1 (Veličković 2018): attention e_ij = LeakyReLU(a^T [Wh_i || Wh_j]). The linear transform before activation means attention is static with respect to the query. GATv2 (Brody 2022): e_ij = a^T · LeakyReLU(W [h_i || h_j]). Dynamic attention — different nodes can attend differently. GATv2 is strictly more expressive.

**Q: How would you build a protein-protein interaction prediction model with GNNs?**
A: (1) Represent each protein as a graph: residues = nodes, edges = contacts within 8Å. (2) Use GNN (EGNN or GAT) to get per-residue embeddings, pool to protein embedding. (3) For interaction: concatenate two protein embeddings, pass through interaction head. (4) Train on BioGRID/STRING PPI data. (5) Hard negative mining: sample proteins with similar sequences but no interaction.

In [ ]:
# --- Imports ---
import torch
import torch.nn as nn

# --- EGNNLayer class definition ---
class EGNNLayer(nn.Module):
    """E(n) Equivariant Graph Neural Network layer (Satorras et al. 2021)."""
    # --- __init__() function ---
    def __init__(self, d_in, d_hidden=64):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(d_in * 2 + 1, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_hidden), nn.SiLU()
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(d_in + d_hidden, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, d_in)
        )
        # Set coord_mlp
        self.coord_mlp = nn.Sequential(
            nn.Linear(d_hidden, d_hidden), nn.SiLU(),
            nn.Linear(d_hidden, 1)
        )

    # --- forward() function ---
    def forward(self, h, x, edge_index):
        src, dst = edge_index
        r_sq = ((x[src] - x[dst])**2).sum(-1, keepdim=True)
        # Combine tensors
        e_input = torch.cat([h[src], h[dst], r_sq], dim=-1)
        m = self.edge_mlp(e_input)  # edge messages

        # Equivariant coordinate update
        x_diff = x[src] - x[dst]
        # Initialize tensor
        x_agg = torch.zeros_like(x)
        x_agg.index_add_(0, dst, x_diff * self.coord_mlp(m))

        # Node update
        m_agg = torch.zeros(h.shape[0], m.shape[-1])
        m_agg.index_add_(0, dst, m)
        # Combine tensors
        h_new = h + self.node_mlp(torch.cat([h, m_agg], dim=-1))
        x_new = x + x_agg

        return h_new, x_new

torch.manual_seed(42)
N, d = 30, 16
h = torch.randn(N, d)
x = torch.randn(N, 3)
edge_index = torch.randint(0, N, (2, 90))

layer = EGNNLayer(d_in=d)
h_new, x_new = layer(h, x, edge_index)
print(f"h: {h.shape} -> {h_new.shape}")
print(f"x: {x.shape} -> {x_new.shape}")
print("EGNN key property: coordinate updates are equivariant to rotations/translations")

---
## GNN Architecture Cheat Sheet

### Architecture Decision Guide

```
What is your task?
|
+-- Molecular property prediction (SMILES/3D)?
|   +-- 2D graph only:        GIN or GINE (+ edge features)
|   +-- 3D coordinates avail: EGNN, SchNet, DimeNet++, EquiformerV2
|   +-- Large-scale (>100k):  GCN, GraphSAGE (scalable)
|
+-- Protein structure?
|   +-- Contact/interaction:  GATv2 (anisotropic, edge features)
|   +-- 3D coordinates:       EGNN, GVP-GNN, SE(3)-Transformer
|   +-- AF3-style:            IPA (Invariant Point Attention) = GNN variant
|
+-- Knowledge graph / link prediction?
|   +-- RotatE, TransE (embedding-based)
|   +-- CompGCN, R-GCN (GNN-based)
|
+-- Citation/social network?
|   +-- GCN or GraphSAGE (large scale, inductive)
|
+-- Long-range dependencies critical?
    +-- Graph Transformer (Graphormer, GraphGPS)
    +-- Or: GNN + virtual node trick
```

### Architecture Comparison Table

| Architecture | Year | Aggregation | Expressivity | Inductive | 3D Support | Best Use Case |
|---|---|---|---|---|---|---|
| GCN | 2017 | Mean (spectral) | < 1-WL | No | No | Citation networks, semi-supervised |
| GraphSAGE | 2017 | Mean/Max/LSTM | < 1-WL | Yes | No | Large-scale inductive (social nets) |
| GAT v1 | 2018 | Attention (static) | < 1-WL | Yes | No | Protein interaction, heterophily |
| GIN | 2019 | Sum + MLP | = 1-WL | Yes | No | Molecular property (best expressive) |
| MPNN | 2017 | Sum + edge feats | < 1-WL | Yes | Partial | Chemistry (QM9), edge-featured graphs |
| GATv2 | 2021 | Attention (dynamic) | = 1-WL | Yes | No | When edge context matters |
| EGNN | 2021 | Sum (equivariant) | Equivariant | Yes | Yes | Drug discovery, docking, MD |
| SE(3)-TF | 2020 | Spherical harm. | Equivariant | Yes | Yes | Exact SO(3), QM properties |
| EquiformerV2 | 2023 | SO(3)-equivariant | Equivariant | Yes | Yes | Catalysis, OC20 SOTA |
| Graphormer | 2021 | Global attention | Beyond 1-WL | Yes | Partial | Molecular property (PCQM4M) |
| GraphGPS | 2022 | GNN + Transformer | Beyond 1-WL | Yes | No | General graphs, SOTA benchmarks |

### Complexity Reference

| Operation | Time | Memory | Notes |
|---|---|---|---|
| GCN layer | O(E * F) | O(N * F) | E = edges, F = features |
| GAT layer | O(E * H * F) | O(N * H * F) | H = heads |
| Graph Transformer | O(N² * F) | O(N² + N*F) | Full attention |
| EGNN layer | O(E * F) | O(N * F + E * F) | Extra coord update |
| Global pooling | O(N * F) | O(B * F) | B = batch size |

### Key Equations Summary

```
GCN:        H' = sigma( D^{-1/2}(A+I)D^{-1/2} H W )

GraphSAGE:  h_i' = sigma( W · concat(h_i, MEAN(h_j : j in N(i))) )

GIN:        h_i' = MLP( (1+eps)*h_i + SUM(h_j : j in N(i)) )

GAT v1:     e_ij = LeakyReLU(a^T [Wh_i||Wh_j])
            h_i' = sigma( SUM_j softmax(e_ij) * W*h_j )

GAT v2:     e_ij = a^T · LeakyReLU(W[h_i||h_j])   <- joint transform first

EGNN:       m_ij = phi_e(h_i, h_j, ||x_i-x_j||^2)
            x_i' = x_i + SUM_j (x_i-x_j) * phi_x(m_ij)
            h_i' = h_i + phi_h(h_i, AGG(m_ij))
```

### WL Expressivity Ladder

```
Expressivity (low to high):

  GCN < GraphSAGE-mean < GAT < GraphSAGE-sum ~ GIN ~ GATv2  [<= 1-WL ceiling]

  To break 1-WL ceiling:
  - Higher-order GNNs (k-WL):       Expensive — O(N^k) complexity
  - Structural features:            Cycle counts, node distances, random walk features
  - Global attention:               Graphormer, GraphGPS (global receptive field)
  - 3D equivariant (with coords):   EGNN, SE(3)-TF (position info adds expressivity)
```

### Common Failure Modes

| Issue | Symptom | Fix |
|---|---|---|
| Over-smoothing | Loss plateaus, all embeddings similar | Residuals, JK-Net, shallow GNN + pooling |
| Over-squashing | Long-range task fails | Graph rewiring, virtual node, Graph Transformer |
| WL limit | Cannot distinguish symmetric molecules | GIN + structural features, 3D coords |
| Scale explosion | GCN loss explodes at depth | Add symmetric normalization to adjacency |
| Static attention | GAT ignores edge context | Switch to GATv2 |
| Not equivariant | Model must be retrained for each orientation | Use EGNN / SE(3)-Transformer |

## Mastery Check

Before moving on, make sure you can:
1. explain why proteins can be represented as graphs
2. distinguish invariance from equivariance
3. describe what messages are passed along edges
4. justify one design choice in the graph construction


---
## 🧪 Real Benchmark Validation — TUDataset

All GNN cells above use random synthetic graphs. Real validation requires benchmarks. Here we use the PROTEINS dataset (1,113 real protein graphs, graph classification task).

If `torch_geometric` is installed, we use actual dataset. Otherwise we simulate the exact same statistics.


In [ ]:
# GNN BENCHMARK ON PROTEIN GRAPH DATASET
# TUDataset PROTEINS: 1,113 protein topology graphs, 2-class (enzyme vs non-enzyme)
# Each graph: residues = nodes, edges = spatial contacts within 7 Å

import torch
# --- Imports ---
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt

# Error handling block
try:
    from torch_geometric.datasets import TUDataset
    from torch_geometric.loader import DataLoader as GDataLoader
    from torch_geometric.nn import GCNConv, GATConv, GINConv, global_mean_pool
    TG_AVAILABLE = True
    print("torch_geometric available — using real PROTEINS dataset")
except ImportError:
    TG_AVAILABLE = False
    print("torch_geometric not installed (pip install torch_geometric)")
    print("Simulating PROTEINS dataset statistics for demonstration")

# ── Simulate PROTEINS-like dataset if torch_geometric unavailable ──
class SimGraph:
    # --- __init__() function ---
    def __init__(self, n_nodes, label, n_node_feats=3):
        self.num_nodes = n_nodes
        # Random edges (average degree ~5, like real protein contact graphs)
        n_edges = min(n_nodes * 5, n_nodes * (n_nodes-1) // 2)
        src = torch.randint(0, n_nodes, (n_edges,))
        dst = torch.randint(0, n_nodes, (n_edges,))
        # Combine tensors
        self.edge_index = torch.stack([
            # Combine tensors
            torch.cat([src, dst]),
            # Combine tensors
            torch.cat([dst, src])
        ])
        self.x = torch.randn(n_nodes, n_node_feats)
        self.y = torch.tensor([label])
        # Initialize tensor
        self.batch = torch.zeros(n_nodes, dtype=torch.long)

# --- simulate_proteins_dataset() function ---
def simulate_proteins_dataset(n=200):
    """Simulate PROTEINS dataset: ~n graphs, mean 39 nodes, 2 classes."""
    np.random.seed(42)
    graphs = []
    # --- Loop over range(n) ---
    for i in range(n):
        n_nodes = max(10, int(np.random.normal(39, 15)))
        label = int(i < n // 2)
        graphs.append(SimGraph(n_nodes, label))
    return graphs

if not TG_AVAILABLE:
    dataset = simulate_proteins_dataset(200)
    n_train = 160
    train_data = dataset[:n_train]
    test_data  = dataset[n_train:]
    n_feats = 3
    print(f"Simulated {len(dataset)} protein graphs")
    print(f"Mean nodes per graph: {np.mean([g.num_nodes for g in dataset]):.1f}")
else:
    dataset = TUDataset(root='/tmp/TUDataset', name='PROTEINS')
    n_train = int(0.8 * len(dataset))
    train_data = dataset[:n_train]
    test_data  = dataset[n_train:]
    n_feats = dataset.num_node_features
    print(f"PROTEINS: {len(dataset)} graphs, {n_feats} node features")
    print(f"Mean nodes per graph: {np.mean([d.num_nodes for d in dataset]):.1f}")
    print(f"Classes: {dataset.num_classes}")

# ── Simple GNN models for benchmarking ──
class GCNNet(nn.Module):
    # --- __init__() function ---
    def __init__(self, n_feats, hidden=64, n_classes=2):
        super().__init__()
        if TG_AVAILABLE:
            # Set conv1
            self.conv1 = GCNConv(n_feats, hidden)
            # Set conv2
            self.conv2 = GCNConv(hidden, hidden)
        else:
            # Fallback: simple MLP (no message passing)
            self.conv1 = nn.Linear(n_feats, hidden)
            # Set conv2
            self.conv2 = nn.Linear(hidden, hidden)
        # Set fc
        self.fc = nn.Linear(hidden, n_classes)
    # --- forward() function ---
    def forward(self, data):
        if TG_AVAILABLE:
            x, ei, batch = data.x, data.edge_index, data.batch
            x = F.relu(self.conv1(x, ei))
            x = F.relu(self.conv2(x, ei))
            x = global_mean_pool(x, batch)
        else:
            x = F.relu(self.conv1(data.x))
            x = F.relu(self.conv2(x))
            x = x.mean(dim=0, keepdim=True)  # graph-level pooling
        return self.fc(x)

# --- evaluate() function ---
def evaluate(model, data_list, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        # --- Loop over data_list ---
        for g in data_list:
            out = model(g)
            target = g.y.view(-1)[:1]
            loss = criterion(out, target)
            total_loss += loss.item()
            correct += (out.argmax(1) == target).sum().item()
            total += 1
    return total_loss / total, correct / total

gcn = GCNNet(n_feats)
opt = torch.optim.Adam(gcn.parameters(), lr=5e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print("\nTraining GCN on protein graphs (10 epochs)...")
t0 = time.time()
# --- Loop over range(10) ---
for epoch in range(10):
    # --- Training step ---
    gcn.train()
    # --- Loop over train_data ---
    for g in train_data:
        opt.zero_grad()
        out = gcn(g)
        loss = criterion(out, g.y.view(-1)[:1])
        # Backpropagate gradients
        loss.backward()
        opt.step()
    if (epoch+1) % 5 == 0:
        tr_loss, tr_acc = evaluate(gcn, train_data, criterion)
        te_loss, te_acc = evaluate(gcn, test_data,  criterion)
        print(f"  Epoch {epoch+1:2d}: train={tr_acc:.3f}, test={te_acc:.3f}")
gcn_time = time.time() - t0

_, gcn_acc = evaluate(gcn, test_data, criterion)
print(f"\nGCN final test acc: {gcn_acc:.3f} ({gcn_time:.1f}s)")
print(f"Expected on real PROTEINS: GCN~0.73, GAT~0.74, GIN~0.76")
print(f"Baseline (majority class): ~0.59")

In [ ]:
# SE(3) EQUIVARIANCE VERIFICATION TEST
# Critical: any GNN claiming equivariance must PASS this test
# If you modify an EGNN and break equivariance, this test catches it

import torch
# --- Imports ---
import numpy as np

# --- rotation_matrix_random() function ---
def rotation_matrix_random():
    """Random 3D rotation matrix via Gram-Schmidt."""
    v1 = torch.randn(3)
    v1 = v1 / v1.norm()
    v2 = torch.randn(3)
    v2 = v2 - (v2 @ v1) * v1
    v2 = v2 / v2.norm()
    v3 = torch.cross(v1, v2)
    # Combine tensors
    return torch.stack([v1, v2, v3], dim=0)

# --- SimpleEGNN class definition ---
class SimpleEGNN(nn.Module):
    """Minimal E(3)-equivariant GNN (simplified EGNN)."""
    # --- __init__() function ---
    def __init__(self, n_feat=8, hidden=32):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(2*n_feat + 1, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(n_feat + hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, n_feat)
        )
        # Set coord_mlp
        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden, 1)
        )

    # --- forward() function ---
    def forward(self, h, x, edges):
        """
        h: node features (N, n_feat)
        x: coordinates (N, 3)
        edges: (2, E) edge list
        Returns updated h, x
        """
        src, dst = edges[0], edges[1]
        # Compute relative distances (INVARIANT to rotation/translation)
        diff = x[src] - x[dst]
        dist2 = (diff ** 2).sum(dim=-1, keepdim=True)

        # Edge messages
        edge_feat = torch.cat([h[src], h[dst], dist2], dim=-1)
        m = self.edge_mlp(edge_feat)

        # Coordinate update: weighted sum of directions (EQUIVARIANT)
        w = self.coord_mlp(m)
        # Initialize tensor
        coord_update = torch.zeros_like(x)
        coord_update.scatter_add_(0, dst.unsqueeze(-1).expand_as(diff),
                                  w * diff / (dist2.sqrt() + 1e-8))

        # Node update
        agg = torch.zeros(h.size(0), m.size(-1))
        agg.scatter_add_(0, dst.unsqueeze(-1).expand(-1, m.size(-1)), m)
        # Combine tensors
        h_new = self.node_mlp(torch.cat([h, agg], dim=-1))
        x_new = x + coord_update

        return h_new, x_new

torch.manual_seed(42)

# Create a small test graph
N = 8
n_feat = 8
model = SimpleEGNN(n_feat)
model.eval()

# Random node features and coordinates
h = torch.randn(N, n_feat)
x = torch.randn(N, 3)
edges = torch.tensor([[0,1,2,3,4,5,6,0,2,4],
                       [1,2,3,4,5,6,0,3,5,7]])

# Apply EGNN on original coordinates
with torch.no_grad():
    h_out, x_out = model(h, x, edges)

# Apply random rotation + translation to input
R = rotation_matrix_random()
t = torch.randn(3)
x_rot = (x @ R.T) + t

# Apply EGNN on rotated coordinates
with torch.no_grad():
    h_rot, x_rot_out = model(h, x_rot, edges)

# EQUIVARIANCE TESTS:
# 1. Node features h should be INVARIANT (same regardless of rotation)
feat_diff = (h_out - h_rot).abs().max().item()

# 2. Output coordinates should be EQUIVARIANT (rotate with the input)
x_expected = (x_out @ R.T) + t
coord_diff = (x_rot_out - x_expected).abs().max().item()

print("SE(3) EQUIVARIANCE VERIFICATION TEST")
print("=" * 50)
print(f"Node feature invariance error:    {feat_diff:.2e}")
print(f"Coordinate equivariance error:    {coord_diff:.2e}")
print()
if feat_diff < 1e-4 and coord_diff < 1e-4:
    print("✓ PASS: Model is SE(3)-equivariant within numerical precision")
else:
    print("✗ FAIL: Model is NOT equivariant!")
    print("  If you modified the EGNN code, this tells you exactly what broke.")
print()
print("WHY THIS TEST MATTERS:")
print("  - Equivariant models generalize better to new protein orientations")
print("  - If equivariance is broken, the model learns rotation-specific patterns")
print("  - This is equivalent to a unit test — run it after any architectural change")
print()
print("APPLY THIS TEST TO:")
print("  - EGNN in module 06 → should pass")
print("  - GCN/GAT (not equivariant) → coord_diff will be large")
print("  - Your fine-tuned version → run before submitting any paper")

In [ ]:
# REAL PROTEIN GRAPH FROM ACTUAL PDB COORDINATES
# Build a residue-level contact graph from real 1UBQ coordinates

import numpy as np
# --- Imports ---
import torch

# Use 1UBQ Cα coordinates (first 20 residues, hardcoded from PDB)
ubq_ca = np.array([
    [1.207, -2.658, -0.000],   # MET 1
    [3.256, -3.929, -2.849],   # GLN 2
    [2.753, -7.528, -3.025],   # ILE 3
    [3.948, -7.920, -6.613],   # PHE 4
    [3.318, -4.607, -7.818],   # VAL 5
    [4.867, -4.153,-11.079],   # LYS 6
    [4.559, -0.440,-11.562],   # THR 7
    [6.462, -0.227, -8.621],   # LEU 8
    [9.843, -1.274, -9.485],   # THR 9
    [10.986,-1.289, -6.024],   # GLY 10
    [10.028, 2.298, -5.266],   # LYS 11
    [11.912, 3.350, -2.435],   # THR 12
    [13.082, 0.018, -1.452],   # ILE 13
    [13.987, 1.009,  1.966],   # THR 14
    [12.685, 4.367,  2.268],   # LEU 15
    [11.619, 5.062,  5.667],   # GLU 16
    [11.950, 8.772,  5.695],   # VAL 17
    [10.534, 9.979,  8.716],   # GLU 18
    [7.129,  8.994,  9.119],   # PRO 19
    [6.148, 11.782, 11.200],   # SER 20
])

# Define residue_names list
residue_names = ['MET','GLN','ILE','PHE','VAL','LYS','THR','LEU','THR','GLY',
                 'LYS','THR','ILE','THR','LEU','GLU','VAL','GLU','PRO','SER']

# One-hot encode amino acid type (simplified: hydrophobic/polar/charged/special)
def aa_to_feature(aa):
    # Define hydrophobic dict/set
    hydrophobic = {'ILE', 'LEU', 'VAL', 'PHE', 'MET'}
    # Define polar dict/set
    polar       = {'THR', 'SER', 'GLN'}
    # Define charged dict/set
    charged     = {'LYS', 'GLU', 'ASP', 'ARG'}
    # Return result
    return [int(aa in hydrophobic), int(aa in polar), int(aa in charged),
            # Call int
            int(aa not in hydrophobic|polar|charged)]  # special (GLY, PRO, etc.)

# Build edges: connect residues within 8 Å (standard protein contact threshold)
CONTACT_THRESHOLD = 8.0
# Compute n via len
n = len(ubq_ca)
edges_src, edges_dst = [], []
# Define edge_features list
edge_features = []

# --- Loop over range(n) ---
for i in range(n):
    # --- Loop over range(i+1, n) ---
    for j in range(i+1, n):
        dist = np.linalg.norm(ubq_ca[i] - ubq_ca[j])
        if dist < CONTACT_THRESHOLD:
            edges_src.extend([i, j])
            edges_dst.extend([j, i])
            edge_features.extend([dist, dist])

node_features = torch.tensor([aa_to_feature(aa) for aa in residue_names], dtype=torch.float)
coords = torch.tensor(ubq_ca, dtype=torch.float)
edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)
edge_attr = torch.tensor(edge_features, dtype=torch.float).unsqueeze(-1)

print("UBIQUITIN PROTEIN GRAPH (1UBQ, first 20 residues)")
print(f"  Nodes: {n} (residues)")
print(f"  Edges: {len(edges_src)} (residue pairs within {CONTACT_THRESHOLD} Å)")
print(f"  Node features: {node_features.shape} (hydrophobic/polar/charged/special)")
print(f"  Edge features: {edge_attr.shape} (Cα-Cα distance)")
print(f"  Average degree: {len(edges_src)/n:.1f} contacts per residue")
print()

# Compute degree distribution
degrees = torch.zeros(n, dtype=torch.long)
# --- Loop over edges_src ---
for s in edges_src:
    degrees[s] += 1

print("Residue contact analysis:")
# --- Loop over range(n) ---
for i in range(n):
    marker = '●' if degrees[i].item() >= 5 else '○'
    print(f"  {marker} {residue_names[i]:3s} {i+1:2d}: "
          f"{degrees[i].item():2d} contacts | "
          f"{'hydrophobic core' if residue_names[i] in {'ILE','LEU','VAL','PHE'} else ''}")

print()
print("KEY OBSERVATIONS:")
print("  Residues with most contacts = protein core (hydrophobic burial)")
print("  Residues with fewest contacts = surface exposed or flexible loops")
print()
print("TO PROCESS A FULL PDB FILE INTO A GRAPH:")
print("  1. Download PDB with Bio.PDB.PDBList")
print("  2. Extract all Cα coordinates with parse_ca_coords() from module 03")
print("  3. Build edges with distance threshold (8 Å standard for residue graphs)")
print("  4. Add node features: AA type, secondary structure (DSSP), conservation score")
print("  5. This graph is the input for all GCN/GAT/EGNN models in this module")

## 🐛 Debug Exercise — Graph Neural Network Message Passing

Find and fix the **3 bugs** in the GNN forward pass below.

**Expected correct output:**
```
Output logits (raw, no activation on final layer): tensor of shape [4, 2]
Values can be negative (valid logits for cross-entropy)
Node 0 and Node 1 have different representations after message passing
```

<details>
<summary>Show answer</summary>

**Bug 1 — Missing self-loops:** Without self-loops, a node's own features are not included
in the aggregation. Fix: `A = A + torch.eye(A.shape[0])` before normalizing.

**Bug 2 — Unnormalized adjacency:** Using raw A without degree normalization makes features
scale with node degree. Fix: compute `D^{-1/2} A D^{-1/2}` where D is the degree matrix.

**Bug 3 — ReLU after final layer:** Applying ReLU to the output layer clips negative logits,
breaking cross-entropy loss. Fix: remove `F.relu` from the final layer output.
</details>


In [ ]:
# DEBUG EXERCISE — GNN message passing, find and fix 3 bugs
import torch
import torch.nn.functional as F

# Simple 4-node graph: 0-1, 1-2, 2-3
A = torch.tensor([
    [0., 1., 0., 0.],
    [1., 0., 1., 0.],
    [0., 1., 0., 1.],
    [0., 0., 1., 0.],
], dtype=torch.float32)

X = torch.tensor([
    [1., 0.],
    [0., 1.],
    [1., 1.],
    [0., 0.],
], dtype=torch.float32)

# Two-layer GCN weights (fixed for reproducibility)
torch.manual_seed(0)
W1 = torch.randn(2, 4)
W2 = torch.randn(4, 2)

def gcn_forward(A, X, W1, W2):
    # BUG 1: missing self-loops — node's own features excluded from aggregation
    # Fix: A_hat = A + torch.eye(A.shape[0])
    A_hat = A     # no self-loops

    # BUG 2: adjacency not normalized — node with more neighbors dominates
    # Fix: D = degree matrix, A_norm = D^{-1/2} @ A_hat @ D^{-1/2}
    A_norm = A_hat    # unnormalized

    # Layer 1
    H1 = F.relu(A_norm @ X @ W1.T)

    # Layer 2
    out = A_norm @ H1 @ W2.T

    # BUG 3: ReLU on final layer clips negative logits — wrong for cross-entropy
    out = F.relu(out)   # should just be: return out

    return out

logits = gcn_forward(A, X, W1, W2)
print("Output logits shape:", logits.shape)
print("Logits:\n", logits)
print("\nNote: all values >= 0 because of buggy final ReLU — valid logits can be negative")
print("Node 0 repr:", logits[0].tolist())
print("Node 1 repr:", logits[1].tolist())


## 🌉 Bridge to Module 07 — AlphaFold3 Core

**The jump:** Module 06 built graph neural networks for proteins. Module 07 is AlphaFold3 — the most complex architecture in the curriculum.

**Before starting Module 07:**
1. ✅ Start with `07/00_af3_zero_to_hero.ipynb` — written specifically for this transition
2. ✅ Make sure you can explain: what is a transformer? what is attention? (Module 05)
3. ✅ The Pairformer in AF3 is a specialised attention over a 2D pair matrix — re-read 06/02 if needed
4. 📖 Optional: watch a short AlphaFold2 overview video on YouTube for context

**Key mindset shift:** AF3 is 100,000+ lines of production code. You are NOT expected to understand every line — focus on the key components.

## Notebook Complete

**You can now:**
- Implement message passing, aggregation, and readout layers from scratch
- Understand SE(3)-equivariance and why it matters for 3D molecular models

**Where this fits:**
- Previous: 01_structure_ml
- Next: 07_alphafold3_core/00_af3_zero_to_hero — Module 07 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes